# Render Remotion (MODO B) · Colab / Kaggle · automático
Ejecuta todo (Ctrl+F9). Sin editar celdas: detecta `colab_*.zip` subido (o, si no hay, clona la rama `colab/<PROYECTO>` del repo público), toma el nombre real del proyecto del zip / `props.json`, instala dependencias, renderiza con **GPU T4** (`--gl=angle` → `webgl` → `swangle`) o **CPU** (`swangle`, `--concurrency=2`) si no hay GPU, aplica `loudnorm=I=-14:TP=-1.5:LRA=11` y descarga `<proyecto>.mp4`.

In [ ]:
# 1) Configuración (todo es opcional: vacío = automático)
PROYECTO = "sabias-conspiraciones-v2"                                  # "" = autodetectar (zip subido / props.json). `pipeline.py --render-mode colab` genera una copia con esto ya rellenado
REPO = "jonisral2211-dev/render-remotion"      # repo público con la rama colab/<PROYECTO> (se usa solo si no hay zip)
CONCURRENCIA_GPU, CONCURRENCIA_CPU = 2, 2      # Colab gratis = 2 vCPU
USAR_NVENC = True                              # re-encode final con h264_nvenc si hay GPU

In [ ]:
# 2) GPU + sistema (Ubuntu 24.04: libasound2 -> libasound2t64) · muestra los errores reales
import subprocess, os, glob, json, shutil, time
def sh(c, check=True):
    r = subprocess.run(c, shell=True, text=True, executable="/bin/bash")
    if check and r.returncode != 0:
        raise RuntimeError(f"Falló ({r.returncode}): {c}")
    return r.returncode == 0
def out(c):
    return subprocess.run(c, shell=True, text=True, capture_output=True, executable="/bin/bash").stdout.strip()
GPU = (out("nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null").splitlines() or [""])[0]
HAY_GPU = bool(GPU)
print("GPU:", GPU or "SIN GPU -> render por CPU (swangle). Para T4: Entorno de ejecución > Cambiar tipo > T4")
SUDO = "" if os.geteuid() == 0 else "sudo "
sh(f"{SUDO}apt-get update -qq")
libs = "ffmpeg libnss3 libdbus-1-3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libpango-1.0-0 libcairo2 libvulkan1 mesa-vulkan-drivers unzip git"
sh(f"{SUDO}DEBIAN_FRONTEND=noninteractive apt-get install -y -qq {libs}")
sh(f"{SUDO}DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2t64 || {SUDO}DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2")
nv = out("node -v 2>/dev/null")
if not nv.startswith(("v20", "v22", "v24")):
    print("Node actual:", nv or "ninguno", "-> instalando Node 20")
    sh(f"curl -fsSL https://deb.nodesource.com/setup_20.x | {SUDO}bash - && {SUDO}apt-get install -y -qq nodejs")
print("node", out("node -v"), "|", out("ffmpeg -version | head -1"))

In [ ]:
# 3) Traer el proyecto: colab_*.zip (autodetectado) > rama git colab/<PROYECTO> > subida manual
BASE = "/content" if os.path.isdir("/content") else "/kaggle/working"
os.chdir(BASE)
zips = sorted(glob.glob(f"{BASE}/colab_*.zip") + glob.glob("/kaggle/input/**/colab_*.zip", recursive=True))
if not zips and not PROYECTO:
    try:
        from google.colab import files; files.upload()
    except ImportError:
        pass
    zips = sorted(glob.glob(f"{BASE}/colab_*.zip"))
shutil.rmtree("proyecto", ignore_errors=True)
if zips:
    z = zips[-1]; print("Zip:", z)
    sh(f'unzip -oq "{z}" -d proyecto')
    if not PROYECTO:
        PROYECTO = os.path.basename(z)[len("colab_"):-4]
elif PROYECTO:
    print("Sin zip: clonando", REPO, "rama colab/" + PROYECTO)
    sh(f"git clone -q --depth 1 --branch colab/{PROYECTO} https://github.com/{REPO}.git proyecto")
else:
    raise RuntimeError("No hay colab_*.zip ni PROYECTO")
props = glob.glob("proyecto/render_jobs/*/props.json")
assert props, "El paquete no trae render_jobs/<proyecto>/props.json"
PROYECTO = os.path.basename(os.path.dirname(props[0]))      # el nombre real sale del props.json
PROPS = os.path.abspath(props[0])
print("Proyecto:", PROYECTO)

In [ ]:
# 4) Remotion + Chromium + verificación de paquetes
os.chdir(f"{BASE}/proyecto/remotion-studio")
sh("npm ci --silent --no-audit --no-fund")
sh("npx remotion browser ensure")
deps = json.load(open("package.json"))["dependencies"]
NEC = ["@fontsource/cinzel", "@fontsource/montserrat", "@fontsource/special-elite", "@remotion/transitions", "@remotion/paths", "@remotion/noise", "lucide-react", "roughjs"]
faltan = [d for d in NEC if d not in deps or not os.path.isdir(f"node_modules/{d}")]
if faltan:
    print("Instalando paquetes que faltan:", faltan)
    sh("npm install --silent --no-audit --no-fund " + " ".join(faltan))
print("Paquetes OK")

In [ ]:
# 5) Render: T4 -> angle > webgl > swangle · sin GPU -> swangle con --concurrency=2
os.makedirs("../salida", exist_ok=True)
orden = ["angle", "webgl", "swangle"] if HAY_GPU else ["swangle"]
conc = CONCURRENCIA_GPU if HAY_GPU else CONCURRENCIA_CPU
t = time.time(); usado = None
for gl in orden:
    if sh(f'npx remotion render src/index.ts Main ../salida/crudo.mp4 --props="{PROPS}" --codec=h264 --gl={gl} --concurrency={conc} --log=info', check=False):
        usado = gl; break
    print(f"--gl={gl} falló, probando el siguiente...")
assert usado, "El render falló con todos los modos GL"
print(f"Render OK: {time.time()-t:.0f} s (gl={usado}, concurrency={conc})")

In [ ]:
# 6) loudnorm + descarga con el nombre exacto del proyecto
final = f"../salida/{PROYECTO}.mp4"
LN = "loudnorm=I=-14:TP=-1.5:LRA=11"
if USAR_NVENC and HAY_GPU and sh("ffmpeg -hide_banner -encoders 2>/dev/null | grep -q h264_nvenc", check=False):
    vcodec = "-c:v h264_nvenc -preset p5 -cq 21"
else:
    vcodec = "-c:v copy"
if not sh(f'ffmpeg -y -loglevel error -i ../salida/crudo.mp4 {vcodec} -af {LN} -c:a aac -b:a 192k {final}', check=False):
    sh(f'ffmpeg -y -loglevel error -i ../salida/crudo.mp4 -c:v copy -af {LN} -c:a aac -b:a 192k {final}')
sh(f"ls -lh {final}; ffprobe -v error -show_entries format=duration -of csv=p=0 {final}")
try:
    from google.colab import files; files.download(os.path.abspath(final))
except ImportError:
    print("Kaggle: el archivo queda en", os.path.abspath(final), "(pestaña Output)")